# 01 — Data Exploration

This notebook explores the MVTec Anomaly Detection dataset to understand:
- Number of images per category and class
- Visual differences between good and defective samples
- Image size distribution
- Class balance analysis
- Defect mask visualization

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

DATA_ROOT = Path("../data/raw")
CATEGORIES = ["bottle", "cable", "transistor"]

plt.rcParams["figure.figsize"] = (12, 6)
sns.set_style("whitegrid")

## 1. Dataset Statistics

Count images per category, split, and defect type.

In [ ]:
records = []

for category in CATEGORIES:
    for split in ["train", "test"]:
        split_dir = DATA_ROOT / category / split
        if not split_dir.exists():
            print(f"WARNING: {split_dir} not found")
            continue
        for defect_dir in sorted(split_dir.iterdir()):
            if not defect_dir.is_dir():
                continue
            n_images = len(list(defect_dir.glob("*.png")))
            label = "good" if defect_dir.name == "good" else "defective"
            records.append({
                "category": category,
                "split": split,
                "defect_type": defect_dir.name,
                "label": label,
                "count": n_images,
            })

df = pd.DataFrame(records)
print("Dataset Overview:")
print(df.to_string(index=False))
print(f"\nTotal images: {df['count'].sum()}")

In [ ]:
# Class distribution per category
fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(5 * len(CATEGORIES), 5))

for i, category in enumerate(CATEGORIES):
    cat_df = df[df["category"] == category]
    pivot = cat_df.pivot_table(index="label", columns="split", values="count", aggfunc="sum", fill_value=0)
    pivot.plot(kind="bar", ax=axes[i], rot=0)
    axes[i].set_title(f"{category.capitalize()} — Class Distribution")
    axes[i].set_xlabel("Label")
    axes[i].set_ylabel("Count")
    axes[i].legend(title="Split")

plt.tight_layout()
plt.show()

## 2. Sample Visualization

Display "good" vs "defective" images side by side for each category.

In [ ]:
for category in CATEGORIES:
    test_dir = DATA_ROOT / category / "test"
    if not test_dir.exists():
        print(f"Skipping {category} — test directory not found")
        continue

    # Get one good and one defective image
    good_dir = test_dir / "good"
    defect_dirs = [d for d in test_dir.iterdir() if d.is_dir() and d.name != "good"]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f"{category.capitalize()} — Good vs Defective Samples", fontsize=14)

    # Top row: good samples
    good_images = sorted(good_dir.glob("*.png"))[:4]
    for j, img_path in enumerate(good_images):
        img = Image.open(img_path)
        axes[0, j].imshow(img)
        axes[0, j].set_title("Good", color="green")
        axes[0, j].axis("off")

    # Bottom row: defective samples (from different defect types)
    for j, defect_dir in enumerate(defect_dirs[:4]):
        imgs = sorted(defect_dir.glob("*.png"))
        if imgs:
            img = Image.open(imgs[0])
            axes[1, j].imshow(img)
            axes[1, j].set_title(f"Defect: {defect_dir.name}", color="red")
        axes[1, j].axis("off")

    # Hide unused axes
    for row in range(2):
        for col in range(len(good_images) if row == 0 else len(defect_dirs[:4]), 4):
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()

## 3. Image Size Distribution

Check if all images have the same resolution.

In [ ]:
sizes = []

for category in CATEGORIES:
    for split in ["train", "test"]:
        split_dir = DATA_ROOT / category / split
        if not split_dir.exists():
            continue
        for img_path in split_dir.rglob("*.png"):
            img = Image.open(img_path)
            sizes.append({
                "category": category,
                "width": img.size[0],
                "height": img.size[1],
            })

size_df = pd.DataFrame(sizes)
print("Image size summary by category:")
print(size_df.groupby("category")[["width", "height"]].describe())

## 4. Defect Mask Visualization

Show defect masks (ground truth) alongside the original defective images.

In [ ]:
for category in CATEGORIES:
    gt_dir = DATA_ROOT / category / "ground_truth"
    test_dir = DATA_ROOT / category / "test"
    if not gt_dir.exists():
        print(f"No ground truth masks found for {category}")
        continue

    defect_types = [d.name for d in gt_dir.iterdir() if d.is_dir()]
    n_types = min(len(defect_types), 4)

    fig, axes = plt.subplots(2, n_types, figsize=(4 * n_types, 8))
    fig.suptitle(f"{category.capitalize()} — Defect Masks", fontsize=14)

    if n_types == 1:
        axes = axes.reshape(2, 1)

    for j, defect_type in enumerate(defect_types[:n_types]):
        # Original defective image
        test_imgs = sorted((test_dir / defect_type).glob("*.png"))
        mask_imgs = sorted((gt_dir / defect_type).glob("*.png"))

        if test_imgs and mask_imgs:
            img = Image.open(test_imgs[0])
            mask = Image.open(mask_imgs[0])

            axes[0, j].imshow(img)
            axes[0, j].set_title(f"{defect_type}")
            axes[0, j].axis("off")

            axes[1, j].imshow(mask, cmap="Reds")
            axes[1, j].set_title("Ground Truth Mask")
            axes[1, j].axis("off")

    plt.tight_layout()
    plt.show()

## 5. Observations

Key findings from the data exploration:

1. **Class imbalance by design:** The training set contains only "good" images. This is intentional — MVTec AD is designed for **anomaly detection**, where the model learns what "normal" looks like and flags deviations.

2. **Image quality:** All images are high-resolution and well-lit under controlled conditions, simulating real factory inspection scenarios.

3. **Defect variety:** Each category has multiple distinct defect types with varying severity — from subtle contamination to obvious structural damage.

4. **Our approach:** We use **PatchCore** — a feature-based anomaly detection method that extracts patch-level features from a pretrained WideResNet-50. It builds a memory bank of normal features and detects anomalies via nearest-neighbor distance. The distance heatmap naturally highlights WHERE the defect is, without requiring any defective training samples.